# Building a stock-analyzing AI agent, by pairing a LangGraph ReAct agent with a tool that fetches stock market data using yfinance to calculate percentage gains or drops from the previous close.

In [19]:
import yfinance as yf
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

load_dotenv()

True

# Define Tool - identify declining stocks

In [20]:
@tool
def get_bluechip_declining_stocks() -> str:
    """
    Fetches market data for major blue-chip stocks and returns the top 5 
    decliners based on percentage loss from yesterday's market close.
    """
    # Sample list of major blue-chip tickers
    blue_chips = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "JPM", "PG", "JNJ", "V", "BRK-B"]
    
    stock_data = []
    for symbol in blue_chips:
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            
            if len(hist) >= 2:
                prev_close = hist['Close'].iloc[-2]
                latest_close = hist['Close'].iloc[-1]
                pct_change = ((latest_close - prev_close) / prev_close) * 100
                
                stock_data.append({
                    "ticker": symbol,
                    "latest_price": round(latest_close, 2),
                    "change_percent": round(pct_change, 2)
                })
        except Exception:\
            continue
    
    # Sort stocks ascending (lowest/most negative percent change first)
    sorted_losers = sorted(stock_data, key=lambda x: x["change_percent"])
    return str(sorted_losers[:5])

# Define Tool - identify gaining stocks

In [21]:
@tool
def get_bluechip_gaining_stocks() -> str:
    """
    Fetches market data for major blue-chip stocks and returns the top 5 
    gainers based on percentage growth from yesterday's market close.
    Use this tool when users ask about stocks at near-highs or profit-taking opportunities.
    """
    blue_chips = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "JPM", "PG", "JNJ", "V", "BRK-B"]
    
    stock_data = []
    for symbol in blue_chips:
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            
            if len(hist) >= 2:
                prev_close = hist['Close'].iloc[-2]
                latest_close = hist['Close'].iloc[-1]
                pct_change = ((latest_close - prev_close) / prev_close) * 100
                
                stock_data.append({
                    "ticker": symbol,
                    "latest_price": round(latest_close, 2),
                    "change_percent": round(pct_change, 2)
                })
        except Exception:
            continue
    
    # Sort descending to get the highest positive percentage changes first
    sorted_gainers = sorted(stock_data, key=lambda x: x["change_percent"], reverse=True)
    return str(sorted_gainers[:5])

In [22]:
# Step 2: Initialize Gemini LLM
model = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it",
    temperature=0.2
)

# Step 3: Define Agent System Instructions & Build Agent
system_prompt = """You are a smart stock market analysis assistant.
When a user asks to buy blue-chip stocks at a dip:
1. Call the `get_bluechip_declining_stocks` tool to fetch the biggest DECLINERS from yesterday's close.
2. Call the `get_bluechip_gaining_stocks` tool to fetch the biggest GAINERS from yesterday's close.
3. Filter and analyze the top 3-5 biggest gainers/losers returned by the tool.
4. Present your analysis clearly, outlining the ticker, price, percent drop, and a brief technical/fundamental observation.
5. Include a brief disclaimer that this is educational information and not formal financial advice."""

agent = create_agent(
    model=model,
    tools=[get_bluechip_declining_stocks, get_bluechip_gaining_stocks],
    system_prompt=system_prompt  # Use 'system_prompt' instead of 'prompt'
)

In [23]:
# Step 4: Run the Agent Query
user_query = "I want to buy a blue chip stock at the dip, tell me the biggest loser of yesterday's market close."

result = agent.invoke({"messages": [("user", user_query)]})

# Output the final AI analysis
print(result["messages"][-1].content)

[{'type': 'thinking', 'thinking': 'The user wants the biggest loser from yesterday\'s close to buy at a dip.\nFrom `get_bluechip_declining_stocks`, the biggest loser is AAPL with a change of -7.35% and a price of 308.91.\nI also have the gainers list (AMZN, GOOGL, MSFT, NVDA, PG) as required by the system instructions.\n\nAnalysis:\n- Biggest Loser: AAPL (Apple Inc.)\n- Price: 308.91\n- Percent Drop: -7.35%\n- Observation: A drop of over 7% for a mega-cap like Apple is significant and often presents a "dip" buying opportunity, provided the fundamentals remain strong.\n\nI will now format the response according to the system instructions.'}, {'type': 'text', 'text': "Based on yesterday's market close, the biggest loser among the blue-chip stocks is **Apple Inc. (AAPL)**.\n\n### Market Analysis: Buying the Dip\n\n**Top Decliner:**\n*   **Ticker:** AAPL\n*   **Price:** $308.91\n*   **Percent Drop:** -7.35%\n*   **Observation:** A drop of over 7% for a company of Apple's size is a signific

In [24]:
response_blocks = result["messages"][-1].content

if isinstance(response_blocks, list):
    for block in response_blocks:
        # Display internal reasoning block
        if block.get("type") == "thinking":
            print("🤔 THINKING PROCESS:\n")
            print(block.get("thinking").strip())
            print("\n" + "─" * 60 + "\n")
            
        # Display final response to user
        elif block.get("type") == "text":
            print("💬 FINAL RESPONSE:\n")
            print(block.get("text").strip())
else:
    # Standard string fallback
    print(response_blocks)

🤔 THINKING PROCESS:

The user wants the biggest loser from yesterday's close to buy at a dip.
From `get_bluechip_declining_stocks`, the biggest loser is AAPL with a change of -7.35% and a price of 308.91.
I also have the gainers list (AMZN, GOOGL, MSFT, NVDA, PG) as required by the system instructions.

Analysis:
- Biggest Loser: AAPL (Apple Inc.)
- Price: 308.91
- Percent Drop: -7.35%
- Observation: A drop of over 7% for a mega-cap like Apple is significant and often presents a "dip" buying opportunity, provided the fundamentals remain strong.

I will now format the response according to the system instructions.

────────────────────────────────────────────────────────────

💬 FINAL RESPONSE:

Based on yesterday's market close, the biggest loser among the blue-chip stocks is **Apple Inc. (AAPL)**.

### Market Analysis: Buying the Dip

**Top Decliner:**
*   **Ticker:** AAPL
*   **Price:** $308.91
*   **Percent Drop:** -7.35%
*   **Observation:** A drop of over 7% for a company of Apple'

In [25]:
# Step 4: Run the Agent Query - for gainers
user_query = "I want to sell a blue chip stock at the dip, tell me the biggest gainer of yesterday's market close."

result = agent.invoke({"messages": [("user", user_query)]})

# Output the final AI analysis
print(result["messages"][-1].content)

[{'type': 'thinking', 'thinking': 'The tool `get_bluechip_gaining_stocks` returned a list of the top 5 gainers.\nThe biggest gainer is AMZN with a change of 15.32%.\n\nThe user\'s request was: "I want to sell a blue chip stock at the dip, tell me the biggest gainer of yesterday\'s market close."\nWait, the user said "sell... at the dip", but then asked for the "biggest gainer". Usually, selling at a peak (after a gain) makes more sense, or buying at a dip. Regardless, the direct question is "tell me the biggest gainer".\n\nI will provide the details for AMZN.\n\nAnalysis:\n- Ticker: AMZN\n- Price: 271.58\n- Percent Gain: 15.32%\n- Observation: A significant jump of over 15% suggests a strong positive catalyst (e.g., earnings beat or major product announcement).\n\nI must also include the disclaimer.'}, {'type': 'text', 'text': "The biggest gainer among blue-chip stocks from yesterday's market close was **Amazon (AMZN)**.\n\n**Analysis:**\n*   **Ticker:** AMZN\n*   **Price:** $271.58\n*

In [26]:
response_blocks = result["messages"][-1].content

if isinstance(response_blocks, list):
    for block in response_blocks:
        # Display internal reasoning block
        if block.get("type") == "thinking":
            print("🤔 THINKING PROCESS:\n")
            print(block.get("thinking").strip())
            print("\n" + "─" * 60 + "\n")
            
        # Display final response to user
        elif block.get("type") == "text":
            print("💬 FINAL RESPONSE:\n")
            print(block.get("text").strip())
else:
    # Standard string fallback
    print(response_blocks)

🤔 THINKING PROCESS:

The tool `get_bluechip_gaining_stocks` returned a list of the top 5 gainers.
The biggest gainer is AMZN with a change of 15.32%.

The user's request was: "I want to sell a blue chip stock at the dip, tell me the biggest gainer of yesterday's market close."
Wait, the user said "sell... at the dip", but then asked for the "biggest gainer". Usually, selling at a peak (after a gain) makes more sense, or buying at a dip. Regardless, the direct question is "tell me the biggest gainer".

I will provide the details for AMZN.

Analysis:
- Ticker: AMZN
- Price: 271.58
- Percent Gain: 15.32%
- Observation: A significant jump of over 15% suggests a strong positive catalyst (e.g., earnings beat or major product announcement).

I must also include the disclaimer.

────────────────────────────────────────────────────────────

💬 FINAL RESPONSE:

The biggest gainer among blue-chip stocks from yesterday's market close was **Amazon (AMZN)**.

**Analysis:**
*   **Ticker:** AMZN
*   **